**Keys ideas:**

1. model equation  
   <img src="./doc/img/model_equation.png" alt="model_equation" width="650" />

2. model inputs  
   <img src="./doc/img/model_inputs.png" alt="model_equation" width="650" />


In [2]:
import pandas as pd
from src.recovery_model import RecoveryModel

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Select a folder for the data to be used
folder = "test_fewer_layers"  # choose between: test_1  / test_2  / test_fewer_layers / Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
# layer_4 = "element"

layer_names = (layer_0, layer_1, layer_2, layer_3)

In [3]:
# This section insures that the structure of the excel files is consistent


metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        # "Layer 4": layer_4,
        "Value": "data",
        "parameterCode": "parameterCode",
        "Year": "year",  # not considered at this stage
        "Scenario": "scenario",  # not considered at this stage
        "Location": "region",  # not considered at this stage
        "UoM": "unit",  # not considered at this stage
    },
    "parameterCode": {
        layer_2: "c-p",  # ! do not change value
        layer_3: "m-c",  # ! do not change value
        # layer_4: "e-m",  # ! do not change value
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Unit": "unit",
        "Year": "year",  # not considered at this stage
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "process": "process",
        "Year": "year",  # not considered at this stage
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        # layer_4: "E*",
    },
}

---


# Recovery model


In [4]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
)

---

# Model variables


In [5]:
model.dims

(7, 3, 4, 3)

In [6]:
model.size

252

In [7]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7
process,,,,,,,
T1,1,-1,-1,0,0,1,0
T2,0,1,0,-1,-1,0,0
T4,0,0,1,1,0,-1,-1


In [8]:
model.lneqs  # the A matrix (it will be properly renamed later)

<252x252 sparse matrix of type '<class 'numpy.float64'>'
	with 135 stored elements in Compressed Sparse Row format>

In [9]:
model.y

<252x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

---

# Model function (solver)


In [10]:
model.solve(aggregate=False, pivot=False).fillna("")

,flow,product,component,material,data
0,F1,P1,,,1000.00
1,F1,P1,C1,,250.00
2,F1,P1,C1,M1,130.00
3,F1,P1,C1,M2,120.00
4,F1,P1,C2,,590.00
...,...,...,...,...,...
65,F7,P1,C2,M2,2.04
66,F7,P1,C3,M1,0.83
67,F7,P1,C3,M2,2.59
68,F7,P2,C2,M1,1.66


In [11]:
model.solve(aggregate=False, pivot=True).fillna("")

flow,product,component,material,F1,F2,F3,F4,F5,F6,F7
0,P1,,,1000.00,,,,,,
1,P1,C1,,250.00,62.50,,,,,
2,P1,C1,M1,130.00,32.50,,,12.68,,
3,P1,C1,M2,120.00,30.00,,,9.60,,
4,P1,C2,,590.00,27.76,26.86,5.55,,11.24,
5,P1,C2,M1,389.40,18.32,17.73,3.66,1.83,7.42,2.08
6,P1,C2,M2,200.60,9.44,9.13,1.89,4.72,3.82,2.04
7,P1,C3,,160.00,,13.04,,,1.83,
8,P1,C3,M1,78.40,,6.39,,,0.89,0.83
9,P1,C3,M2,81.60,,6.65,,,0.93,2.59


---

# Mass balance


In [3]:
folder = "test_2"
mass_balance = pd.read_csv(f"consolidation/{folder}_solution_mass_balance.csv", index_col=0)
mass_balance.fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
0,P1,,,,T1,1000.00,,,,,,,,1000.00
1,P1,C1,,,T1,250.00,-62.50,,,,,,,187.50
2,P1,C1,M1,,T1,130.00,-32.50,,,0.00,,,,97.50
3,P1,C1,M1,E1,T1,91.00,-22.75,,,0.00,,,0.00,68.25
4,P1,C1,M1,E2,T1,39.00,-9.75,,,0.00,,,,29.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,P2,C2,M1,E1,T3,0.00,0.00,0.00,0.00,0.97,0.00,0.76,-0.28,1.45
119,P2,C2,M1,E2,T3,0.00,0.00,0.00,0.00,1.14,0.00,0.89,,2.04
120,P2,C2,M2,,T3,0.00,0.00,0.00,0.00,1.31,0.00,0.36,,1.67
121,P2,C2,M2,E1,T3,0.00,0.00,0.00,0.00,0.65,0.00,0.18,,0.83


In [15]:
impossible_rows = mass_balance["mass_balance"] < 0
mass_balance[impossible_rows].fillna("")

,product,component,material,element,process,WEEE_generatedComplementaryExported,WEEE_2RM_thermal2Smelter_other,WEEE_mechRec2_chem,WEEE_generatedComplementaryUndocumented,WEEE_2RM_mechRec1Smelter_other,...,WEEE_2RM_chem2Smelter_other,WEEE_generatedComplementaryMetalScrap,WEEE_collected,WEEE_generatedDedicated,WEEE_mechRec2_thermal,WEEE_mechRec1_mechRec2,WEEE_chem_thermal,WEEE_categ_mechRec2,WEEE_categ_mechRec1,mass_balance
8022,WEEE_Cat1,,,,WEEE_generated,-135511.97,,,-830010.82,,...,,-355718.92,-948583.79,1693899.63,,,,0.00,0.00,-1507570.67
8023,WEEE_Cat1,ComponentShadowWEEE,,,WEEE_generated,-1162.72,,,-7121.63,0.00,...,,-3052.13,-8139.01,14533.95,,0.00,,0.00,0.00,-12935.21
8024,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,,WEEE_generated,-79.03,,,-484.07,0.00,...,,-207.46,-553.23,987.90,,0.00,,0.00,0.00,-879.23
8025,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,Ag,WEEE_generated,-11.68,,,-71.53,0.00,...,,-30.66,-81.75,145.98,,0.00,,0.00,0.00,-129.92
8026,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,As,WEEE_generated,-0.21,,,-1.28,0.00,...,,-0.55,-1.46,2.60,,0.00,,0.00,0.00,-2.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29563,WEEE_Cat4b,passiveJunctionBox,CuAndCuAlloys,Zr,dismantling,0.00,,,0.00,-2.62,...,,0.00,0.00,0.00,,-2.99,,0.00,5.24,-0.37
29564,WEEE_Cat4b,passiveJunctionBox,otherOrUndefinedMaterials,,dismantling,0.00,,,0.00,-1.04,...,,0.00,0.00,0.00,,-1.18,,0.00,2.07,-0.14
29565,WEEE_Cat4b,passiveJunctionBox,otherOrUndefinedMaterials,Ag,dismantling,0.00,,,0.00,-1.04,...,,0.00,0.00,0.00,,-1.18,,0.00,2.07,-0.14
29566,WEEE_Cat4b,passiveJunctionBox,plastics,,dismantling,0.00,,,0.00,-3.71,...,,0.00,0.00,0.00,,-4.23,,0.00,7.42,-0.52
